In [1]:
import lightning as L
from pathlib import Path
import yaml


from src.models import get_model
from src.config import TrainConfig
from src.datamodules import SimpleDataModule
import warnings
import rasterio

warnings.filterwarnings(
    "ignore",
    category=rasterio.errors.NotGeoreferencedWarning,
)

config_ = "config_files/v3/segnet_loveda_v3.yaml"

with open(config_, "r") as f:
    config = yaml.safe_load(f)

config_path = Path(config_)
VERSION = config_path.parent.name


cfg = TrainConfig(**config)

L.seed_everything(42)

model = get_model(cfg.model, **cfg.model_kwargs.model_dump())

Seed set to 42


In [2]:
dm = SimpleDataModule(
    dataset_name=cfg.dataset_name,
    batch_size=cfg.batch_size,
    val_test_batch_size=cfg.val_test_batch_size,
)
print(dm.train_image_glob)
dm.setup(stage="fit")
len(dm.train_data)

LoveDA/train/images/*.png


2522

In [3]:
from src.segmentors import SimpleSegmentor

segmentor = SimpleSegmentor(
    model,
    n_classes=cfg.n_classes,
    criterion=cfg.get_loss(),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
    hf_model=cfg.model,
)

In [ ]:
dl = next(iter(dm.val_dataloader()))
dl["image"].shape, dl["mask"].shape

dl["mask"].shape

torch.Size([8, 1024, 1024])

: 

In [ ]:
segmentor.validation_step(dl, 0)

In [ ]:
segmentor.training_step(dl, 0)

tensor(0.2095, grad_fn=<MeanBackward0>)